In [58]:
import sqlite3
import warnings
from sqlalchemy.exc import SAWarning

# Mute the specific SQLAlchemy decimal warning
warnings.filterwarnings("ignore", category=SAWarning)

SOURCE = "../data/Chinook_Sqlite.sqlite"
TARGET = "../data/music_sales.sqlite"

src = sqlite3.connect(SOURCE)
src.execute(f"ATTACH DATABASE '{TARGET}' AS out")

src.execute("DROP TABLE IF EXISTS out.music_sales")
src.execute("""
    CREATE TABLE out.music_sales AS
    SELECT
        il.InvoiceLineId                            AS id,
        g.Name                                       AS genre,
        ar.Name                                      AS artist,
        i.BillingCountry                             AS country,
        CAST(strftime('%Y', i.InvoiceDate) AS INT)   AS year,
        t.Name                                       AS track_name,
        il.UnitPrice                                 AS unit_price,
        il.Quantity                                  AS quantity,
        il.UnitPrice * il.Quantity                   AS extended_price
    FROM InvoiceLine il
    JOIN Invoice  i  ON i.InvoiceId = il.InvoiceId
    JOIN Track    t  ON t.TrackId = il.TrackId
    LEFT JOIN Album  al ON al.AlbumId = t.AlbumId
    LEFT JOIN Artist ar ON ar.ArtistId = al.ArtistId
    LEFT JOIN Genre  g  ON g.GenreId = t.GenreId
""")
src.commit()

n = src.execute("SELECT COUNT(*) FROM out.music_sales").fetchone()[0]
print(f"done -- {TARGET} created with {n} rows in table music_sales")

done -- ../data/music_sales.sqlite created with 2240 rows in table music_sales


In [59]:
from cubes import Workspace

workspace=Workspace(load_base_model=False)
workspace.register_default_store("sql", url="sqlite:///../data/music_sales.sqlite")
workspace.import_model("../models/model.json")

browser=workspace.browser("music_sales")
cube=workspace.cube("music_sales")

In [60]:
from cubes import Cell
print("1. MEMBERS -- what genres, countries, years exist?")

genre_dim=cube.dimension("genre")
country_dim=cube.dimension("country")
year_dim=cube.dimension("year")

genre_members=sorted(list(browser.members(Cell(cube),genre_dim)), key=lambda m : m["genre"])
country_members=sorted(list(browser.members(Cell(cube),country_dim)), key=lambda m : m["country"])
year_members=sorted(list(browser.members(Cell(cube),year_dim)), key=lambda m : m["year"])

print(f"{len(genre_members)} genres, e.g {genre_members[:5]}")
print(f"{len(country_members)} countries, e.g {country_members[:5]}")
print(f"{len(year_members)} year values, e.g {year_members[:5]}")

1. MEMBERS -- what genres, countries, years exist?
24 genres, e.g [{'genre': 'Alternative'}, {'genre': 'Alternative & Punk'}, {'genre': 'Blues'}, {'genre': 'Bossa Nova'}, {'genre': 'Classical'}]
24 countries, e.g [{'country': 'Argentina'}, {'country': 'Australia'}, {'country': 'Austria'}, {'country': 'Belgium'}, {'country': 'Brazil'}]
5 year values, e.g [{'year': 2021}, {'year': 2022}, {'year': 2023}, {'year': 2024}, {'year': 2025}]


In [61]:
genre_dim = cube.dimension("genre")
genres = sorted(m["genre"] for m in browser.members(Cell(cube), genre_dim))
print(f"  {len(genres)} genres, e.g.: {genres[:5]}")

  24 genres, e.g.: ['Alternative', 'Alternative & Punk', 'Blues', 'Bossa Nova', 'Classical']


In [62]:
print(genres)
print(genre_members)

['Alternative', 'Alternative & Punk', 'Blues', 'Bossa Nova', 'Classical', 'Comedy', 'Drama', 'Easy Listening', 'Electronica/Dance', 'Heavy Metal', 'Hip Hop/Rap', 'Jazz', 'Latin', 'Metal', 'Pop', 'R&B/Soul', 'Reggae', 'Rock', 'Rock And Roll', 'Sci Fi & Fantasy', 'Science Fiction', 'Soundtrack', 'TV Shows', 'World']
[{'genre': 'Alternative'}, {'genre': 'Alternative & Punk'}, {'genre': 'Blues'}, {'genre': 'Bossa Nova'}, {'genre': 'Classical'}, {'genre': 'Comedy'}, {'genre': 'Drama'}, {'genre': 'Easy Listening'}, {'genre': 'Electronica/Dance'}, {'genre': 'Heavy Metal'}, {'genre': 'Hip Hop/Rap'}, {'genre': 'Jazz'}, {'genre': 'Latin'}, {'genre': 'Metal'}, {'genre': 'Pop'}, {'genre': 'R&B/Soul'}, {'genre': 'Reggae'}, {'genre': 'Rock'}, {'genre': 'Rock And Roll'}, {'genre': 'Sci Fi & Fantasy'}, {'genre': 'Science Fiction'}, {'genre': 'Soundtrack'}, {'genre': 'TV Shows'}, {'genre': 'World'}]


In [63]:
print("2. AGGREGATE -- all totals")

2. AGGREGATE -- all totals


In [64]:
result=browser.aggregate()
print(f"total revenue: ${result.summary['revenue_sum']}M")
print(f"quantity of sold tracks: {result.summary['quantity_sum']}")
print(f"max revenue: {result.summary['max_revenue']}")
print(f"min revenue: {result.summary['revenue_min']}")

total revenue: $2328.599999999957M
quantity of sold tracks: 2240
max revenue: 1.99
min revenue: 0.99


In [65]:
print("3. DRILLDOWN -- revenue by genre (top 5)")

3. DRILLDOWN -- revenue by genre (top 5)


In [66]:
result=browser.aggregate(drilldown=["genre"])

list_genres_sorted=sorted(list(result.cells), key=lambda m : m["revenue_sum"],reverse=True)

for row in list_genres_sorted[:5]:
    print(f"genre: {row['genre']}, revenue_sum: {row['revenue_sum']}")

genre: Rock, revenue_sum: 826.6500000000061
genre: Latin, revenue_sum: 382.14000000000203
genre: Metal, revenue_sum: 261.3600000000009
genre: Alternative & Punk, revenue_sum: 241.56000000000074
genre: TV Shows, revenue_sum: 93.52999999999994


In [67]:
print("4. SLICE -- only genre = Rock")

4. SLICE -- only genre = Rock


In [68]:
from cubes import PointCut

cut=PointCut("genre",["Rock"])

cell=Cell(cube,cuts=[cut])

result=browser.aggregate(cell,drilldown=["genre"])

for row in list(result.cells):
    print(f"genre: {row['genre']}, revenue_sum: {row['revenue_sum']}, quantity: {row['quantity_sum']}")

genre: Rock, revenue_sum: 826.6500000000061, quantity: 835


In [69]:
print("4b. SLICE -- multiple dimensions at once (Rock, USA, 2010)")

4b. SLICE -- multiple dimensions at once (Rock, USA, 2010)


In [70]:
cuts = [
    PointCut("genre", ["Rock"]),
    PointCut("country",["USA"]),
    PointCut("year",[2022])
]
cell= Cell(cube, cuts)

result = browser.aggregate(cell)
print(f"ROCK/USA revenue: {result.summary['revenue_sum']},quantity: {result.summary['quantity_sum']}")

ROCK/USA revenue: 30.68999999999998,quantity: 31


In [71]:
print("5. FACTS -- detail rows behind 'Rock, USA,2010'")

5. FACTS -- detail rows behind 'Rock, USA,2010'


In [72]:
result=browser.facts(cell)
print("details about ROCK tracks sold in 2022 in the USA")
for row in result:
    print(f" artist: {row['artist']},track name: {row['track_name']} quantity: {row['quantity']},extended price: {row['extended_price']}")

details about ROCK tracks sold in 2022 in the USA
 artist: U2,track name: Desire quantity: 1,extended price: 0.99
 artist: U2,track name: When Love Comes To Town quantity: 1,extended price: 0.99
 artist: U2,track name: Angel Of Harlem quantity: 1,extended price: 0.99
 artist: U2,track name: Sunday Bloody Sunday quantity: 1,extended price: 0.99
 artist: U2,track name: New Year's Day quantity: 1,extended price: 0.99
 artist: U2,track name: Drowning Man quantity: 1,extended price: 0.99
 artist: U2,track name: Two Hearts Beat As One quantity: 1,extended price: 0.99
 artist: U2,track name: Surrender quantity: 1,extended price: 0.99
 artist: U2,track name: Zooropa quantity: 1,extended price: 0.99
 artist: Deep Purple,track name: You Fool No One quantity: 1,extended price: 0.99
 artist: Santana,track name: Maria Maria quantity: 1,extended price: 0.99
 artist: Deep Purple,track name: Going Down / Highway Star quantity: 1,extended price: 0.99
 artist: Faith No More,track name: Star A.D. quantit

In [73]:
print("6. MULTI-DIMENSIONAL -- Genre x Year (first 8 rows)")

6. MULTI-DIMENSIONAL -- Genre x Year (first 8 rows)


In [75]:
result=browser.aggregate(drilldown=['genre','year'])

current_genre=None

for row in list(result.cells)[:8]:
    genre=row['genre']
    if genre!=current_genre:
        print(f"genre:{genre}")
        current_genre=genre
    
    print(f"year: {row['year']}, revenue_sum: {row['revenue_sum']}, quantity: {row['quantity_sum']}")

genre:Alternative
year: 2022, revenue_sum: 5.94, quantity: 6
year: 2023, revenue_sum: 3.96, quantity: 4
year: 2024, revenue_sum: 3.96, quantity: 4
genre:Alternative & Punk
year: 2021, revenue_sum: 62.37000000000004, quantity: 63
year: 2022, revenue_sum: 39.599999999999994, quantity: 40
year: 2023, revenue_sum: 45.540000000000006, quantity: 46
year: 2024, revenue_sum: 38.60999999999999, quantity: 39
year: 2025, revenue_sum: 55.440000000000026, quantity: 56
